## Get my dataset

In [1]:
import pandas as pd
data = pd.read_csv("ragas-data.csv")
print("Questions to be evaluated: ", len(data))

Questions to be evaluated:  5


In [2]:
data

,question,ground_truth,answer
0,What is the address of the Dandora dumpsite?,The address of the Dandora dumpsite is Dandora...,The address of the Dandora dumpsite is Dandora...
1,What are the opening hours of the Nairobi City...,The Nairobi City Council recycling center oper...,The Nairobi City Council recycling center oper...
2,Do I need to make an appointment for the recyc...,"No, you do not need to make an appointment to ...","No, you do not need to make an appointment to ..."
3,What are the opening hours of the Nairobi Nati...,The Nairobi National Library is open on weekda...,The Nairobi National Library is open on weekda...
4,What is the address of the Nairobi National Li...,The address of the Nairobi National Library is...,The address of the Nairobi National Library is...


## Evaluation

In [3]:
import ragas
import ragas.metrics as metrics
from langchain_openai.chat_models import AzureChatOpenAI
from langchain_openai.embeddings import AzureOpenAIEmbeddings
import datasets


azure_model = AzureChatOpenAI(azure_deployment="gpt-4o", api_version="2025-02-01-preview")
azure_embeddings = AzureOpenAIEmbeddings(azure_deployment="text-embedding-ada-002", api_version="2025-02-01-preview")

/workspaces/presentation_protect-your-llm-solution/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [4]:
ragas_dataset = {
    "question" : data["question"].to_list(),
    "ground_truth" : data["ground_truth"].to_list(),
    "answer" : data["answer"].to_list()
}
ds = datasets.Dataset.from_dict(ragas_dataset)

result = ragas.evaluate(
    dataset= ds,
    metrics=[metrics.answer_correctness],
    llm=azure_model,
    embeddings=azure_embeddings,
    raise_exceptions=False
)
merged_df = pd.merge(data, pd.DataFrame(result.scores), left_index=True, right_index=True)

Evaluating: 100%|██████████| 5/5 [00:13<00:00,  2.77s/it]


In [5]:
result

{'answer_correctness': 0.9243}

In [6]:
pd.DataFrame(result.scores)

,answer_correctness
0,1.000000
1,0.621633
2,1.000000
3,1.000000
4,1.000000
